<a href="https://colab.research.google.com/github/Thangtho/du_an_quet_tai_lieu_k74/blob/main/D%E1%BB%B1_%C3%A1n_t%E1%BB%B1_%C4%91%E1%BB%99ng_qu%C3%A9t_t%C3%A0i_li%E1%BB%87u_k74.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Thiết lập và Import thư viện

In [ ]:

!apt-get install tesseract-ocr tesseract-ocr-vie -y -qq
!pip install pytesseract img2pdf tqdm -q

import os, cv2, re, numpy as np
import pytesseract, img2pdf
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from google.
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Các hàm hỗ trợ

In [ ]:
def sap_xep_diem_khung(cac_diem):
    hcn  = np.zeros((4, 2), dtype='float32')
    tong = cac_diem.sum(axis=1)
    hieu = cac_diem[:, 1] - cac_diem[:, 0]
    hcn[0] = cac_diem[np.argmin(tong)] # Trái trên
    hcn[2] = cac_diem[np.argmax(tong)] # phải dưới
    hcn[1] = cac_diem[np.argmin(hieu)] # phải trên
    hcn[3] = cac_diem[np.argmax(hieu)] # trái dưới
    return hcn

def bien_doi_phoi_canh(anh_goc, cac_diem):

    khung = sap_xep_diem_khung(cac_diem)
    (tt, tp, pd, td) = khung
    rong = max(int(np.linalg.norm(pd-td)), int(np.linalg.norm(tp-tt)))
    cao  = max(int(np.linalg.norm(tp-pd)), int(np.linalg.norm(tt-td)))
    if rong <= 0 or cao <= 0:
        return anh_goc.copy()
    dich = np.array([[0,0],[rong-1,0],[rong-1,cao-1],[0,cao-1]], dtype='float32')
    M    = cv2.getPerspectiveTransform(khung, dich) #matrix3x3
    return cv2.warpPerspective(anh_goc, M, (rong, cao))

def lam_sach_tai_lieu(anh_xam):
    gamma = 0.5
    anh_gamma = np.array(255 * (anh_xam / 255) ** gamma, dtype='uint8')

    anh_median = cv2.medianBlur(anh_gamma, 3)

    anh_loc = cv2.GaussianBlur(anh_median, (5, 5), 0)
    sach = cv2.adaptiveThreshold(
        anh_loc, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        51, 11
    )
    return sach

def chong_nghieng(anh_sach):

    try:
        _, nhi_phan = cv2.threshold(anh_sach, 127, 255, cv2.THRESH_BINARY_INV)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (20, 3))
        gian_no = cv2.dilate(nhi_phan, kernel, iterations=1)

        bien = cv2.Canny(gian_no, 50, 150)
        duong_thang = cv2.HoughLinesP(bien, 1, np.pi/180, 100, minLineLength=80, maxLineGap=10)

        goc_nghieng = []
        if duong_thang is not None:
            for d in duong_thang:
                x1, y1, x2, y2 = d[0]
                goc = np.degrees(np.arctan2(y2 - y1, x2 - x1))
                if -30 <= goc <= 30:
                    goc_nghieng.append(goc)

        if goc_nghieng:
            goc_chuan = np.median(goc_nghieng)
            if abs(goc_chuan) > 0.5:
                (h, w) = anh_sach.shape[:2]
                tam = (w // 2, h // 2)

                M = cv2.getRotationMatrix2D(tam, goc_chuan, 1.0)
                anh_sach = cv2.warpAffine(anh_sach, M, (w, h), flags=cv2.INTER_CUBIC,
                                          borderMode=cv2.BORDER_CONSTANT, borderValue=255)
    except Exception as e:
        pass
    return anh_sach

## Vòng lặp xử lý chính

In [ ]:
DUONG_DAN_VAO = '/content/drive/MyDrive/thu_vien_anh'
DUONG_DAN_RA  = '/content/drive/MyDrive/XLA_2025.2/ket_qua_2'
os.makedirs(DUONG_DAN_RA, exist_ok=True)

tap_tin_anh = sorted([
    f for f in os.listdir(DUONG_DAN_VAO)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
])

danh_sach_pdf = []
bao_cao       = []

print(f'\n>>> BẮT ĐẦU XỬ LÝ {len(tap_tin_anh)} TÀI LIỆU\n')

for stt, ten_file in enumerate(tqdm(tap_tin_anh, desc='Quét tài liệu', unit='ảnh')):
    duong_dan_anh = os.path.join(DUONG_DAN_VAO, ten_file)
    ten_luu       = os.path.splitext(ten_file)[0]

    try:

        anh_nguon = cv2.imread(duong_dan_anh)
        if anh_nguon is None:
            bao_cao.append((ten_file, 'Bỏ qua', 'File hỏng'))
            continue

        anh_xam = cv2.cvtColor(anh_nguon, cv2.COLOR_BGR2GRAY)

        ti_le_thu_nho = 500.0 / anh_xam.shape[1]
        kthuoc_moi = (500, int(anh_xam.shape[0] * ti_le_thu_nho))
        anh_do_mo = cv2.resize(anh_xam, kthuoc_moi)
        do_net = cv2.Laplacian(cv2.GaussianBlur(anh_do_mo, (3, 3), 0), cv2.CV_64F).var()

        if do_net < 100:
            bao_cao.append((ten_file, 'Bỏ qua', f'Mờ nhòe ({do_net:.0f})'))
            continue

        mo_bien    = cv2.GaussianBlur(anh_xam, (5, 5), 0)
        bien_canny = cv2.Canny(mo_bien, 50, 150)
        cac_duong_bao, _ = cv2.findContours(bien_canny, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cac_duong_bao = sorted(cac_duong_bao, key=cv2.contourArea, reverse=True)[:5]

        anh_da_cat = anh_xam
        dt_anh = anh_xam.shape[0] * anh_xam.shape[1]

        for d in cac_duong_bao:
            chu_vi  = cv2.arcLength(d, True)
            da_giac = cv2.approxPolyDP(d, 0.02 * chu_vi, True)
            dien_tich = cv2.contourArea(da_giac)
            if len(da_giac) == 4 and (0.2 * dt_anh < dien_tich < 0.95 * dt_anh):
                anh_da_cat = bien_doi_phoi_canh(anh_xam, da_giac.reshape(4, 2).astype('float32'))
                break
        try:

            osd_text = pytesseract.image_to_osd(anh_da_cat, config='--psm 0 -c min_characters_to_try=10')
            goc = int(re.search(r'(?<=Rotate: )\d+', osd_text).group(0))
            conf = float(re.search(r'(?<=Orientation confidence: )[\d.]+', osd_text).group(0))
            if conf > 1.5:
                if goc == 90: anh_da_cat = cv2.rotate(anh_da_cat, cv2.ROTATE_90_CLOCKWISE)
                elif goc == 180: anh_da_cat = cv2.rotate(anh_da_cat, cv2.ROTATE_180)
                elif goc == 270: anh_da_cat = cv2.rotate(anh_da_cat, cv2.ROTATE_90_COUNTERCLOCKWISE)
        except: pass

        anh_sach_se = chong_nghieng(lam_sach_tai_lieu(anh_d
        anh_to_ocr = cv2.resize(anh_sach_se,
        van_ban_ocr = pytesseract.image_to_string(anh_to_ocr, lang='vie+eng', config='--oem 3 --psm 3').strip()
        cac_chu_that_su = re.findall(r'[a-zA-Z0-9À-ỹ]', van_ban_ocr)

        if len(cac_chu_that_su) < 10:
            bao_cao.append((ten_file, 'Bỏ qua', f'Không đủ chữ ({len(cac_chu_that_su)})'))
            continue

        if stt < 3:
            fig, axs = plt.subplots(1, 4, figsize=(20, 5))
            axs[0].imshow(cv2.cvtColor(anh_nguon, cv2.COLOR_BGR2RGB)); axs[0].set_title("1. Ảnh Gốc")
            axs[1].imshow(bien_canny, cmap='gray'); axs[1].set_title("2. Canny Edge Detection")
            axs[2].imshow(anh_da_cat, cmap='gray'); axs[2].set_title("3. Cắt Khung & Chuẩn Hóa")
            axs[3].imshow(anh_sach_se, cmap='gray'); axs[3].set_title("4. Làm Sạch & Chống Nghiêng")
            for ax in axs: ax.axis('off')
            plt.show()

        duong_dan_png = os.path.join(DUONG_DAN_RA, f'{ten_luu}_quet.png')
        cv2.imwrite(duong_dan_png, anh_sach_se)
        danh_sach_pdf.append(duong_dan_png)

        with open(os.path.join(DUONG_DAN_RA, f'{ten_luu}.txt'), 'w', encoding='utf-8') as f:
            f.write(van_ban_ocr)

        bao_cao.append((ten_file, 'OK', f'Chữ: {len(cac_chu_that_su)} | Nét: {do_net:.0f}'))

    except Exception as e:

        bao_cao.append((ten_file, 'Lỗi', str(e)[:35]))

print('\n>>> XỬ LÝ XONG!\n')

## Báo cáo kết quả và tạo PDF

In [ ]:
print('='*65)
print('  BÁO CÁO KẾT QUẢ')
print('='*65)
print(f"  {'Tên file':<30} {'Trạng thái':<12} {'Ghi chú'}")
print('-'*65)
for r in bao_cao:
    print(f'  {r[0]:<30} {r[1]:<12} {r[2]}')
print('='*65)

ok = sum(1 for r in bao_cao if r[1] == 'OK')
print(f'  TỔNG KẾT: {ok}/{len(tap_tin_anh)} ảnh thành công')

if danh_sach_pdf:
    file_pdf = os.path.join(DUONG_DAN_RA, 'Du_an_K74_HOAN_CHINH.pdf')
    try:
        with open(file_pdf, 'wb') as f:
            f.write(img2pdf.convert(sorted(danh_sach_pdf)))
        print(f'  Gộp {len(danh_sach_pdf)} trang vào file Du_an_K74_HOAN_CHINH.pdf thành công!')
    except Exception as e:
        print(f'  Lỗi gộp PDF: {e}')